# 01 Construct Corpus - Jupyter Notebook
This notebook is responsible for downaloading the train and validations splits of the HotpotQA dataset, exploring its contents and making a Pandas DataFrame containing the formated and deduplicated Corpus.

In [1]:
from datasets import load_dataset
from collections import Counter
import pandas as pd
from pathlib import Path
import json

In [2]:
def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / ".git").exists():
            return path
    raise FileNotFoundError(f"No .git folder found above {start}")
        
ROOT = find_repo_root()
DATA_DIR = ROOT / "data"
CORPUS_PATH = DATA_DIR / "corpus" / "pooled_corpus.parquet"
CORPUS_PATH.parent.mkdir(parents=True, exist_ok=True)
STATS_PATH = ROOT / "data" / "corpus" / "corpus_stats.json"

print(ROOT) 

/mnt/data/Programing/NLP_cw


In [3]:
hotpot_ds = load_dataset("hotpotqa/hotpot_qa", "distractor")

In [4]:
train = pd.DataFrame(hotpot_ds["train"])
validation = pd.DataFrame(hotpot_ds["validation"])

print(f"Training Dataset size: {len(train)} rows")
print(f"Validation Dataset size: {len(validation)} rows")

Training Dataset size: 90447 rows
Validation Dataset size: 7405 rows


In [5]:
# 2. Counts by 'type' for each split
train_type_counts = Counter(train["type"])
validation_type_counts = Counter(validation["type"])

train_level_counts = Counter(train["level"])
validation_level_counts = Counter(validation["level"])

print(f"Train Dataset Question Type Counts: {train_type_counts}")
print(f"Train Dataset Question Level Counts: {train_level_counts}\n")

print(f"Validation Dataset Question Type Counts: {validation_type_counts}")
print(f"Validation Dataset Question Level Counts: {validation_level_counts}")

Train Dataset Question Type Counts: Counter({'bridge': 72991, 'comparison': 17456})
Train Dataset Question Level Counts: Counter({'medium': 56814, 'easy': 17972, 'hard': 15661})

Validation Dataset Question Type Counts: Counter({'bridge': 5918, 'comparison': 1487})
Validation Dataset Question Level Counts: Counter({'hard': 7405})


### Training Dataset Examples


In [6]:
print(f"Columns Names: {train.columns.tolist()}\n")
print(f"Full Row Content: \n{train.iloc[0]}\n")
print(f"Context Entries: {train.iloc[0].context}\n")
print(f"Suporting Facts Entry: {train.iloc[0].supporting_facts}")

Columns Names: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context']

Full Row Content: 
id                                           5a7a06935542990198eaf050
question            Which magazine was started first Arthur's Maga...
answer                                              Arthur's Magazine
type                                                       comparison
level                                                          medium
supporting_facts    {'title': ['Arthur's Magazine', 'First for Wom...
context             {'title': ['Radio City (Indian radio station)'...
Name: 0, dtype: object

Context Entries: {'title': ['Radio City (Indian radio station)', 'History of Albanian football', 'Echosmith', "Women's colleges in the Southern United States", 'First Arthur County Courthouse and Jail', "Arthur's Magazine", '2014–15 Ukrainian Hockey Championship', 'First for Women', 'Freeway Complex Fire', 'William Rast'], 'sentences': [["Radio City is India's first priva

### Build Corpus

In [7]:
print(f"Titles List: {train.iloc[0].context["title"]}")
print(f"Sentences: {train.iloc[0].context["sentences"]}")


def make_corpus(dfs, split_names):
    corpus = []

    for df, split in zip(dfs, split_names):
        table = list()
        for i in range(len(df)):

            titles = df.iloc[i].context["title"]
            sentences = df.iloc[i].context["sentences"]

            for title, sentence_list in zip(titles, sentences):
                text = "".join(sentence_list)
                row = (split, title, sentence_list, text)
                table.append(row)

        split_corpus = pd.DataFrame(table, columns=["split", "title", "sentence_list", "text"])
        corpus.append(split_corpus)

    full_corpus = pd.concat(corpus, ignore_index=True)
    return full_corpus


def make_corpus_dedups(dfs, split_names):
    corpus = []
    sizes = {"raw_per_split":{"train": 0, "validation": 0}, 
             "deduped_per_split":{"train": 0, "validation": 0},
             "full_corpus": 0, "overlap": 0, "pooled_corpus": 0 }

    for df, split in zip(dfs, split_names):
        table = list()
        for i in range(len(df)):

            titles = df.iloc[i].context["title"]
            sentences = df.iloc[i].context["sentences"]

            for title, sentence_list in zip(titles, sentences):
                text = "".join(sentence_list)
                row = (split, title, sentence_list, text)
                table.append(row)

        split_corpus = pd.DataFrame(table, columns=["split", "title", "sentence_list", "text"])
        sizes["raw_per_split"][split] = len(split_corpus)

        deduped_split_corpus = split_corpus.drop_duplicates(subset=["title", "text"])
        sizes["deduped_per_split"][split] = len(deduped_split_corpus)

        corpus.append(deduped_split_corpus)

    full_corpus = pd.concat(corpus, ignore_index=True)
    sizes["full_corpus"] = len(full_corpus)

    split_overlap = full_corpus.duplicated(subset=["title", "text"]).sum()
    sizes["overlap"] = int(split_overlap)

    pooled_corpus = full_corpus.drop_duplicates(subset=["title", "text"]).reset_index(drop=True)
    sizes["pooled_corpus"] = len(pooled_corpus)

    return pooled_corpus, sizes


Titles List: ['Radio City (Indian radio station)', 'History of Albanian football', 'Echosmith', "Women's colleges in the Southern United States", 'First Arthur County Courthouse and Jail', "Arthur's Magazine", '2014–15 Ukrainian Hockey Championship', 'First for Women', 'Freeway Complex Fire', 'William Rast']
Sentences: [["Radio City is India's first private FM radio station and was started on 3 July 2001.", ' It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).', ' It plays Hindi, English and regional songs.', ' It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007.', ' Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features.', ' The Radio station currently plays a mix of Hindi and 

In [8]:
dfs = [train, validation]
split_names = ["train", "validation"]

corpus, sizes = make_corpus_dedups(dfs, split_names)

# Save Pooled Corpus as Parquet file
corpus.to_parquet(CORPUS_PATH, index=False)

# Save the sizes
with open(STATS_PATH, "w") as f:
    json.dump(sizes, f, indent=2)